# ADHD-EEG Robust Benchmark

Thin orchestration notebook over `src/`. This notebook does not reimplement
any pipeline logic — every function it calls lives in `src/` and is unit-
audited in `AUDIT_REPORT.md`. Run `python tests/test_synthetic_pipeline.py`
first if you have not already, to confirm the pipeline mechanics are sound
in your environment before spending real compute here.

**Before running this notebook on real data**, set `ADHD_EEG_DATA_ROOT`
(see `README.md`) and make sure a GPU is available — see README
"Computational cost" for why.


In [ ]:
import os, sys, time
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))
from src import config, data, models, evaluation, statistics, visualization, sanity_checks, coral as coral_mod, explainability

USE_SYNTHETIC = not os.path.isdir(config.DATA_ROOT)
print("DATA_ROOT:", config.DATA_ROOT, " exists:", not USE_SYNTHETIC)
if USE_SYNTHETIC:
    print("Real dataset not found in this environment -- falling back to the SYNTHETIC self-test "
          "generator so this notebook still runs end to end. Any numbers produced below are NOT "
          "real results in that case. See AUDIT_REPORT.md and FINAL_REPORT.md.")


## Phase 1-2: Load data, reused leakage-free normalization

In [ ]:
if USE_SYNTHETIC:
    X, y, groups, file_epoch_idx = data.make_synthetic_dataset(seed=0)
    fs_a, n_time_a = 128, config.EPOCH_LEN_A if False else 768
else:
    X, y, groups, file_epoch_idx = data.load_dataset()
    fs_a, n_time_a = config.FS_ASSUMED_HZ, config.EPOCH_LEN_A

print("X:", X.shape, " y:", y.shape, " unique subjects:", len(set(groups)))
X_norm = data.normalize_all(X)


## Phase 4-5: Nested + repeated subject-independent CV

For every outer fold: inner GroupKFold on outer-train subjects selects the
CNN's training length via early stopping on inner-validation subjects only;
the final model is retrained on all outer-train subjects for that many
epochs (no validation split); outer-test subjects are touched exactly once,
for prediction. See `AUDIT_REPORT.md` finding L1 for why this differs from
the original notebooks, and `src/evaluation.py` for the implementation.

In [ ]:
all_predictions, all_fold_records, all_coral_rows = [], [], []
seeds = config.RANDOM_SEEDS

for i, seed in enumerate(seeds):
    preds, folds, coral_rows = evaluation.run_nested_repeat(
        "CNN", models.build_cnn, X_norm, y, groups,
        outer_folds=config.OUTER_FOLDS, inner_folds=config.INNER_FOLDS, seed=seed, repeat_id=i,
        preprocessing_label=f"{fs_a}Hz", fs=fs_a, n_timesamples=n_time_a,
        max_epochs=config.MAX_EPOCHS, batch_size=config.BATCH_SIZE,
        patience=config.EARLY_STOPPING_PATIENCE, learning_rate=config.LEARNING_RATE,
        run_classifiers=True, run_coral=True,
    )
    all_predictions.extend(preds); all_fold_records.extend(folds); all_coral_rows.extend(coral_rows)
    print(f"repeat {i} (seed={seed}) done: {len(folds)} outer folds")

predictions_df = pd.DataFrame(all_predictions)
coral_df = pd.DataFrame(all_coral_rows)
predictions_df.to_csv(os.path.join(config.RESULTS_DIR, "predictions.csv"), index=False)
predictions_df.head()


## Phase 6: Alternative EEG architectures (EEGNet, ShallowConvNet, DeepConvNet)

Same nested protocol, same preprocessing, same training budget -- no architecture gets a head start.

In [ ]:
for arch_name, build_fn in models.ARCHITECTURE_BUILDERS.items():
    if arch_name == "CNN":
        continue
    for i, seed in enumerate(seeds):
        preds, folds, _ = evaluation.run_nested_repeat(
            arch_name, build_fn, X_norm, y, groups,
            outer_folds=config.OUTER_FOLDS, inner_folds=config.INNER_FOLDS, seed=seed, repeat_id=i,
            preprocessing_label=f"{fs_a}Hz", fs=fs_a, n_timesamples=n_time_a,
            max_epochs=config.MAX_EPOCHS, batch_size=config.BATCH_SIZE,
            patience=config.EARLY_STOPPING_PATIENCE, learning_rate=config.LEARNING_RATE,
            run_classifiers=False, run_coral=False,
        )
        all_predictions.extend(preds); all_fold_records.extend(folds)
        print(arch_name, "repeat", i, "done")

predictions_df = pd.DataFrame(all_predictions)
predictions_df.to_csv(os.path.join(config.RESULTS_DIR, "predictions.csv"), index=False)


## Phase 8-9: Subject-level performance + bootstrap 95% CI

Subject-level aggregation is the primary clinical evaluation (Phase 8). Bootstrap CIs resample subjects, never epochs (Phase 9).

In [ ]:
subject_tables = {}
for model_name, grp in predictions_df.groupby("model"):
    sdf = evaluation.aggregate_subject_level(grp["subject_id"], grp["y_true"], grp["y_proba"])
    subject_tables[model_name] = sdf
    m = evaluation.compute_metrics(sdf["y_true"], sdf["pred_mean_proba"], sdf["mean_proba"])
    print(f"{model_name:20s} subject-level acc={m['accuracy']:.3f} bal_acc={m['balanced_accuracy']:.3f} "
          f"sens={m['sensitivity']:.3f} spec={m['specificity']:.3f} mcc={m['mcc']:.3f} auc={m['auc']:.3f}")

def acc_fn(df):
    from sklearn.metrics import accuracy_score
    return accuracy_score(df["y_true"], df["pred_mean_proba"])

if "CNN" in subject_tables:
    ci = statistics.bootstrap_subject_level_ci(subject_tables["CNN"], acc_fn, n_boot=config.N_BOOTSTRAP, seed=1)
    print("CNN subject-level accuracy 95% CI:", ci)


## Phase 10-11: Statistical comparisons (Holm-corrected, with effect size) and balanced metrics

In [ ]:
per_fold_metric = {}
for model_name, grp in predictions_df.groupby("model"):
    vals = []
    for (rep, fold), g in grp.groupby(["repeat", "outer_fold"]):
        vals.append(evaluation.compute_metrics(g["y_true"], g["y_pred"])["balanced_accuracy"])
    per_fold_metric[model_name] = vals

cmp_df = statistics.compare_all_models(per_fold_metric, metric_name="balanced_accuracy", reference_model="CNN")
cmp_df.to_csv(os.path.join(config.TABLES_DIR, "table6_model_comparison.csv"), index=False)
cmp_df


## Phase 12: Ablation study

In [ ]:
ablation_rows = []
for ablation_name, cfg in models.ABLATION_CONFIGS.items():
    build_fn = lambda n_channels, n_timesamples, fs, _cfg=cfg: models.build_cnn_ablation(
        n_channels=n_channels, n_timesamples=n_timesamples, fs=fs, **_cfg)
    preds, folds, _ = evaluation.run_nested_repeat(
        f"CNN_{ablation_name}", build_fn, X_norm, y, groups,
        outer_folds=config.OUTER_FOLDS, inner_folds=config.INNER_FOLDS, seed=seeds[0], repeat_id=0,
        preprocessing_label=f"{fs_a}Hz", fs=fs_a, n_timesamples=n_time_a,
        max_epochs=config.MAX_EPOCHS, batch_size=config.BATCH_SIZE,
        patience=config.EARLY_STOPPING_PATIENCE, learning_rate=config.LEARNING_RATE,
        run_classifiers=True, run_coral=False,
    )
    p_df = pd.DataFrame(preds)
    m = evaluation.compute_metrics(p_df["y_true"], p_df["y_pred"], p_df["y_proba"])
    ablation_rows.append({"ablation": ablation_name, "accuracy_mean": m["accuracy"], "accuracy_sd": 0.0, **m})
    all_predictions.extend(preds); all_fold_records.extend(folds)

ablation_df = pd.DataFrame(ablation_rows)
ablation_df.to_csv(os.path.join(config.TABLES_DIR, "table8_ablation.csv"), index=False)
ablation_df


## Phase 3: 128 Hz vs 128->512 Hz interpolation sensitivity experiment

**Not** a reproduction of the original 512 Hz acquisition -- see `AUDIT_REPORT.md` §5. Labeled `128to512_interp` everywhere.

In [ ]:
target_fs = config.FS_INTERP_HZ if not USE_SYNTHETIC else fs_a * 4
X_interp = data.resample_pipeline_b(X, orig_fs=fs_a, target_fs=target_fs)
X_interp_norm = data.normalize_all(X_interp)

preds_b, folds_b, _ = evaluation.run_nested_repeat(
    "CNN", models.build_cnn, X_interp_norm, y, groups,
    outer_folds=config.OUTER_FOLDS, inner_folds=config.INNER_FOLDS, seed=seeds[0], repeat_id=0,
    preprocessing_label="128to512_interp", fs=target_fs, n_timesamples=X_interp_norm.shape[2],
    max_epochs=config.MAX_EPOCHS, batch_size=config.BATCH_SIZE,
    patience=config.EARLY_STOPPING_PATIENCE, learning_rate=config.LEARNING_RATE,
    run_classifiers=True, run_coral=False,
)
all_predictions.extend(preds_b); all_fold_records.extend(folds_b)

df_a = pd.DataFrame([r for r in all_predictions if r["model"]=="CNN" and r["preprocessing"]==f"{fs_a}Hz"])
df_b = pd.DataFrame(preds_b)
m_a = evaluation.compute_metrics(df_a["y_true"], df_a["y_pred"], df_a["y_proba"])
m_b = evaluation.compute_metrics(df_b["y_true"], df_b["y_pred"], df_b["y_proba"])
pd.DataFrame([{"pipeline": "128Hz", **m_a}, {"pipeline": "128to512_interp", **m_b}]).to_csv(
    os.path.join(config.TABLES_DIR, "table7_resolution_sensitivity.csv"), index=False)


## Phase 13: CORAL, explicitly labeled transductive

In [ ]:
print(coral_df[["repeat","outer_fold","cov_distance_before","cov_distance_after",
                 "no_coral_accuracy","with_coral_accuracy","no_coral_auc","with_coral_auc"]])
coral_df.to_csv(os.path.join(config.TABLES_DIR, "table9_coral_vs_no_coral.csv"), index=False)


## Phase 15: Explainability (Integrated Gradients)

No causal claims. Reported with cross-sample mean ± SD to show attribution stability, per the brief's explicit instruction not to hide instability.

In [ ]:
model_final, _ = models.build_cnn(n_channels=X_norm.shape[1], n_timesamples=X_norm.shape[2], fs=fs_a)
import tensorflow as tf
model_final.compile(optimizer=tf.keras.optimizers.Adam(config.LEARNING_RATE), loss="binary_crossentropy")
model_final.fit(X_norm, y, epochs=min(config.MAX_EPOCHS, 20), batch_size=config.BATCH_SIZE, verbose=0)

ig = explainability.batch_channel_temporal_importance(model_final, X_norm, max_samples=30, steps=50, seed=0)
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10,3))
ax.bar(range(len(ig["channel_importance_mean"])), ig["channel_importance_mean"],
       yerr=ig["channel_importance_sd"])
ax.set_xlabel("channel index"); ax.set_ylabel("mean |IG attribution|")
ax.set_title("Channel importance (mean +/- SD across sampled epochs)")
fig.savefig(os.path.join(config.FIGURES_DIR, "fig11_channel_importance.png"), dpi=200, bbox_inches="tight")


## Phase 20: Sanity checks

In [ ]:
predictions_df = pd.DataFrame(all_predictions)
predictions_df.to_csv(os.path.join(config.RESULTS_DIR, "predictions.csv"), index=False)
report = sanity_checks.run_sanity_checks(all_fold_records, predictions_df)
assert report["passed"].all(), "sanity checks failed -- see printed report"


## Phase 19: Figures

In [ ]:
visualization.fig3_nested_cv_schematic(config.OUTER_FOLDS, config.INNER_FOLDS)

summary_rows = []
for model_name, vals in per_fold_metric.items():
    s = statistics.summarize_repeats(vals)
    summary_rows.append({"model": model_name, **{f"balanced_accuracy_{k}": v for k, v in s.items()}})
summary_df = pd.DataFrame(summary_rows)
visualization.fig4_model_comparison_ci(summary_df.rename(columns={
    "balanced_accuracy_mean":"balanced_accuracy_mean","balanced_accuracy_ci_low":"balanced_accuracy_ci_low",
    "balanced_accuracy_ci_high":"balanced_accuracy_ci_high"}), metric="balanced_accuracy")

visualization.fig5_subject_level_roc(subject_tables)
visualization.fig6_subject_confusion_matrices(subject_tables)
visualization.fig7_repeated_cv_distribution(per_fold_metric, metric_name="balanced_accuracy")
if len(coral_df) > 0:
    visualization.fig10_coral_covariance(coral_df)
print("Figures written to", config.FIGURES_DIR)
